# 标量查询（Scalar Query）

**用途**：教学演示 - 使用标量字段进行条件筛选

**核心概念**：
- 标量字段 vs 向量字段
- 过滤表达式（Filter Expression）
- 标量查询的应用场景

## 运行前检查

1. 已安装依赖：pip install -r ../requirements.txt
2. 已完成 01_milvus_basics/ 的学习
3. 确保 Milvus 服务可用（会自动创建测试数据）

In [1]:
from rag_examples.milvus_config import MILVUS_URI
from dotenv import load_dotenv
load_dotenv()

True

## 理解标量查询

### 定义
标量查询 = 基于标量字段（非向量字段）的条件筛选
类似于传统数据库的 SQL WHERE 子句

### 生活化比喻
标量查询 = "Excel 表格的筛选功能"

### 标量字段类型
- INT64：整数（如浏览量、价格）
- FLOAT：浮点数（如价格、评分）
- VARCHAR：字符串（如类别、标题）
- BOOL：布尔值（如是否发布）

### 与向量检索的区别
- 标量查询：精确匹配或范围筛选（如 category=="AI" AND views>1000）
- 向量检索：语义相似度匹配（如"机器学习" → 找相关文档）

### 应用场景
1. 按类别筛选：category == "AI"
2. 按时间范围：create_time > 1700000000
3. 按数值范围：price < 100 AND views > 1000
4. 组合条件：category == "AI" AND views > 500

## 示例 1: 准备测试数据

In [2]:
def prepare_test_collection():
    from pymilvus import MilvusClient, FieldSchema, CollectionSchema, DataType
    import random

    random.seed(42)

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "scalar_query_demo"

    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    fields = [
        FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
        FieldSchema(name="content", dtype=DataType.VARCHAR, max_length=1000),
        FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=256),
        FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="zhangliang", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="views", dtype=DataType.INT64),
        FieldSchema(name="price", dtype=DataType.FLOAT),
        FieldSchema(name="is_published", dtype=DataType.BOOL),
        FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=768),
    ]
    schema = CollectionSchema(fields=fields)

    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        metric_type="COSINE"
    )

    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="embedding",
        index_type="FLAT",
        metric_type="COSINE"
    )
    client.create_index(
        collection_name=collection_name,
        index_params=index_params
    )

    documents = [
        {"content": "人工智能简介", "title": "AI 入门", "category": "AI", "zhangliang": "深度学习", "views": 1000, "price": 0.0, "is_published": True},
        {"content": "机器学习基础", "title": "ML 教程", "category": "AI", "zhangliang": "强化学习", "views": 800, "price": 99.0, "is_published": True},
        {"content": "深度学习入门", "title": "DL 指南", "category": "AI", "zhangliang": "卷积神经网络", "views": 1200, "price": 129.0, "is_published": True},
        {"content": "自然语言处理", "title": "NLP 详解", "category": "AI", "zhangliang": "Transformer", "views": 600, "price": 89.0, "is_published": False},
        {"content": "计算机视觉", "title": "CV 应用", "category": "AI", "zhangliang": "目标检测", "views": 500, "price": 79.0, "is_published": True},
        {"content": "产品设计方法", "title": "产品指南", "category": "Product", "zhangliang": "用户调研", "views": 300, "price": 0.0, "is_published": True},
        {"content": "用户体验优化", "title": "UX 技巧", "category": "Product", "zhangliang": "交互设计", "views": 450, "price": 59.0, "is_published": True},
        {"content": "市场营销策略", "title": "营销实战", "category": "Marketing", "zhangliang": "数字营销", "views": 200, "price": 149.0, "is_published": False},
        {"content": "数据分析方法", "title": "分析教程", "category": "Data", "zhangliang": "统计分析", "views": 700, "price": 0.0, "is_published": True},
        {"content": "Python 编程", "title": "编程入门", "category": "Programming", "zhangliang": "面向对象", "views": 1500, "price": 79.0, "is_published": True},
    ]

    def mock_embedding():
        return [random.random() for _ in range(768)]

    data_to_insert = []
    for doc in documents:
        data_to_insert.append({
            "content": doc["content"],
            "title": doc["title"],
            "category": doc["category"],
            "zhangliang": doc["zhangliang"],
            "views": doc["views"],
            "price": doc["price"],
            "is_published": doc["is_published"],
            "embedding": mock_embedding()
        })

    client.insert(collection_name, data_to_insert)
    client.flush(collection_name)

    print(f"✓ 已创建测试集合：{collection_name}")
    print(f"  插入文档数：{len(documents)}")

    return client, collection_name

In [3]:
client, collection_name = prepare_test_collection()

✓ 已创建测试集合：scalar_query_demo
  插入文档数：10


## 示例 2: 基础标量查询

In [4]:
def basic_scalar_query(client, collection_name):
    """
    1. 查询类别为'AI'的文档：
        过滤条件：category == 'AI'
    2. 查询浏览量>800 的文档：
        过滤条件：views > 800
    3. 查询已发布的文档：
        过滤条件：is_published == True
    """
    print("=" * 60)
    print("示例 2: 基础标量查询（单条件）")
    print("=" * 60)

    client.load_collection(collection_name)

    print("\n1. 查询类别为'AI'的文档：")
    print("   过滤条件：category == 'AI'")
    print("-" * 50)

    results = client.query(
        collection_name=collection_name,
        filter="category == 'AI'",
        output_fields=["title", "category", "views"],
        limit=10
    )

    for res in results:
        print(f"  - {res['title']} | 类别：{res['category']} | 浏览：{res['views']}")

    print("\n2. 查询浏览量>800 的文档：")
    print("   过滤条件：views > 800")
    print("-" * 50)

    results = client.query(
        collection_name=collection_name,
        filter="views > 800",
        output_fields=["title", "views"],
        limit=10
    )

    for res in results:
        print(f"  - {res['title']} | 浏览：{res['views']}")

    print("\n3. 查询已发布的文档：")
    print("   过滤条件：is_published == True")
    print("-" * 50)

    results = client.query(
        collection_name=collection_name,
        filter="is_published == True",
        output_fields=["title", "is_published"],
        limit=10
    )

    for res in results:
        status = "✓" if res['is_published'] else "✗"
        print(f"  {status} {res['title']}")

In [5]:
basic_scalar_query(client, collection_name)

示例 2: 基础标量查询（单条件）

1. 查询类别为'AI'的文档：
   过滤条件：category == 'AI'
--------------------------------------------------
  - AI 入门 | 类别：AI | 浏览：1000
  - ML 教程 | 类别：AI | 浏览：800
  - DL 指南 | 类别：AI | 浏览：1200
  - NLP 详解 | 类别：AI | 浏览：600
  - CV 应用 | 类别：AI | 浏览：500

2. 查询浏览量>800 的文档：
   过滤条件：views > 800
--------------------------------------------------
  - AI 入门 | 浏览：1000
  - DL 指南 | 浏览：1200
  - 编程入门 | 浏览：1500

3. 查询已发布的文档：
   过滤条件：is_published == True
--------------------------------------------------
  ✓ AI 入门
  ✓ ML 教程
  ✓ DL 指南
  ✓ CV 应用
  ✓ 产品指南
  ✓ UX 技巧
  ✓ 分析教程
  ✓ 编程入门


## 示例 3: 组合条件查询

In [6]:
def compound_scalar_query(client, collection_name):
    print()
    print("=" * 60)
    print("示例 3: 组合条件查询")
    print("=" * 60)

    print("\n1. 查询 AI 类别且浏览量>600 的文档：")
    print("   过滤条件：category == 'AI' and views > 600")
    print("-" * 50)

    results = client.query(
        collection_name=collection_name,
        filter="category == 'AI' and views > 600",
        output_fields=["title", "category", "views"],
        limit=10
    )

    for res in results:
        print(f"  - {res['title']} | 类别：{res['category']} | 浏览：{res['views']}")

    print("\n2. 查询 AI 或 Product 类别的文档：")
    print("   过滤条件：category == 'AI' or category == 'Product'")
    print("-" * 50)

    results = client.query(
        collection_name=collection_name,
        filter="category == 'AI' or category == 'Product'",
        output_fields=["title", "category"],
        limit=10
    )

    for res in results:
        print(f"  - {res['title']} | 类别：{res['category']}")

    print("\n3. 查询特定类别的文档：")
    print("   过滤条件：category in ['AI', 'Data']")
    print("-" * 50)

    results = client.query(
        collection_name=collection_name,
        filter="category in ['AI', 'Data']",
        output_fields=["title", "category"],
        limit=10
    )

    for res in results:
        print(f"  - {res['title']} | 类别：{res['category']}")

    print("\n4. 查询已发布且 (浏览量>500 或价格=0) 的文档：")
    print("   过滤条件：is_published == True and (views > 500 or price == 0)")
    print("-" * 50)

    results = client.query(
        collection_name=collection_name,
        filter="is_published == True and (views > 500 or price == 0)",
        output_fields=["title", "views", "price"],
        limit=10
    )

    for res in results:
        print(f"  - {res['title']} | 浏览：{res['views']} | 价格：{res['price']}")

In [7]:
compound_scalar_query(client, collection_name)


示例 3: 组合条件查询

1. 查询 AI 类别且浏览量>600 的文档：
   过滤条件：category == 'AI' and views > 600
--------------------------------------------------
  - AI 入门 | 类别：AI | 浏览：1000
  - ML 教程 | 类别：AI | 浏览：800
  - DL 指南 | 类别：AI | 浏览：1200

2. 查询 AI 或 Product 类别的文档：
   过滤条件：category == 'AI' or category == 'Product'
--------------------------------------------------
  - AI 入门 | 类别：AI
  - ML 教程 | 类别：AI
  - DL 指南 | 类别：AI
  - NLP 详解 | 类别：AI
  - CV 应用 | 类别：AI
  - 产品指南 | 类别：Product
  - UX 技巧 | 类别：Product

3. 查询特定类别的文档：
   过滤条件：category in ['AI', 'Data']
--------------------------------------------------
  - AI 入门 | 类别：AI
  - ML 教程 | 类别：AI
  - DL 指南 | 类别：AI
  - NLP 详解 | 类别：AI
  - CV 应用 | 类别：AI
  - 分析教程 | 类别：Data

4. 查询已发布且 (浏览量>500 或价格=0) 的文档：
   过滤条件：is_published == True and (views > 500 or price == 0)
--------------------------------------------------
  - AI 入门 | 浏览：1000 | 价格：0.0
  - ML 教程 | 浏览：800 | 价格：99.0
  - DL 指南 | 浏览：1200 | 价格：129.0
  - 产品指南 | 浏览：300 | 价格：0.0
  - 分析教程 | 浏览：700 | 价格：0.0
  - 编程入门 | 浏览：1500 | 价格

## 示例 4: 范围查询

In [10]:
def range_query(client, collection_name):
    print()
    print("=" * 60)
    print("示例 4: 范围查询")
    print("=" * 60)

    print("\n1. 查询浏览量在 500-1000 之间的文档：")
    print("   过滤条件：views >= 500 and views <= 1000")
    print("-" * 50)

    results = client.query(
        collection_name=collection_name,
        filter="views >= 500 and views <= 1000",
        output_fields=["title", "views"],
        limit=10
    )

    for res in results:
        print(f"  - {res['title']} | 浏览：{res['views']}")

    print("\n2. 查询价格<100 的文档：")
    print("   过滤条件：price < 100")
    print("-" * 50)

    results = client.query(
        collection_name=collection_name,
        filter="price < 100",
        output_fields=["title", "price"],
        limit=10
    )

    for res in results:
        print(f"  - {res['title']} | 价格：¥{res['price']}")

In [11]:
range_query(client, collection_name)

2026-04-30 10:25:16,611 [ERROR][_log_rpc_error]: RPC error: [query], <MilvusException: (code=1100, message=failed to create query plan: cannot parse expression: views between 500 and 1000, error: invalid expression: views between 500 and 1000: invalid parameter)>, <elapsed:47.0ms>
Traceback:
Traceback (most recent call last):
  File "C:\沃林AI课程\AI0226_0309课程\wolin_learn-master\.venv\Lib\site-packages\pymilvus\decorators.py", line 404, in handler
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\沃林AI课程\AI0226_0309课程\wolin_learn-master\.venv\Lib\site-packages\pymilvus\decorators.py", line 451, in handler
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\沃林AI课程\AI0226_0309课程\wolin_learn-master\.venv\Lib\site-packages\pymilvus\decorators.py", line 342, in handler
    raise e from e
  File "C:\沃林AI课程\AI0226_0309课程\wolin_learn-master\.venv\Lib\site-packages\pymilvus\decorators.py", line 305, in handler
    return func(*args, **


示例 4: 范围查询

1. 查询浏览量在 500-1000 之间的文档：
   过滤条件：views >= 500 and views <= 1000
--------------------------------------------------


MilvusException: <MilvusException: (code=1100, message=failed to create query plan: cannot parse expression: views between 500 and 1000, error: invalid expression: views between 500 and 1000: invalid parameter)>

## 示例 5: 标量 + 向量混合查询

In [12]:
def scalar_plus_vector_search(client, collection_name):
    print()
    print("=" * 60)
    print("示例 5: 标量 + 向量混合查询")
    print("=" * 60)

    import random
    random.seed(42)
    query_vector = [random.random() for _ in range(768)]

    print("\n1. 向量检索（无过滤）：")
    print("   查询向量：768 维随机向量")
    print("   返回 Top-3 最相似的文档")
    print("-" * 50)

    results = client.search(
        collection_name=collection_name,
        data=[query_vector],
        limit=3,
        output_fields=["title", "category", "views"]
    )

    print('全部结果：', results)

    print("检索结果：")
    for hit in results[0]:
        print(f"  [相似度：{hit['distance']:.4f}] {hit['entity']['title']} | 类别：{hit['entity']['category']}")

    print("\n2. 向量检索 + 标量过滤：")
    print("   查询向量：768 维随机向量")
    print("   过滤条件：只返回 AI 类别的文档")
    print("-" * 50)

    results = client.search(
        collection_name=collection_name,
        # 向量字段检索
        data=[query_vector],
        limit=3,
        output_fields=["title", "category", "views"],
        # 标量过滤
        filter="category == 'AI'"
    )

    print("检索结果（仅 AI 类别）：")
    for hit in results[0]:
        print(f"  [相似度：{hit['distance']:.4f}] {hit['entity']['title']} | 类别：{hit['entity']['category']}")

In [13]:
scalar_plus_vector_search(client, collection_name)


示例 5: 标量 + 向量混合查询

1. 向量检索（无过滤）：
   查询向量：768 维随机向量
   返回 Top-3 最相似的文档
--------------------------------------------------
全部结果： data: [[{'id': 465448300126374471, 'distance': 0.9999998807907104, 'entity': {'title': 'AI 入门', 'category': 'AI', 'views': 1000}}, {'id': 465448300126374477, 'distance': 0.7603273987770081, 'entity': {'title': 'UX 技巧', 'category': 'Product', 'views': 450}}, {'id': 465448300126374472, 'distance': 0.7596487998962402, 'entity': {'title': 'ML 教程', 'category': 'AI', 'views': 800}}]]
检索结果：
  [相似度：1.0000] AI 入门 | 类别：AI
  [相似度：0.7603] UX 技巧 | 类别：Product
  [相似度：0.7596] ML 教程 | 类别：AI

2. 向量检索 + 标量过滤：
   查询向量：768 维随机向量
   过滤条件：只返回 AI 类别的文档
--------------------------------------------------
检索结果（仅 AI 类别）：
  [相似度：1.0000] AI 入门 | 类别：AI
  [相似度：0.7596] ML 教程 | 类别：AI
  [相似度：0.7553] CV 应用 | 类别：AI


## 示例 6: 标量查询最佳实践

In [ ]:
def scalar_query_best_practices():
    print()
    print("=" * 60)
    print("示例 6: 标量查询最佳实践")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ 标量查询最佳实践                                        │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ 1. 过滤表达式语法                                        │
│    - 等于：field == 'value'                             │
│    - 不等于：field != 'value'                           │
│    - 大于/小于：field > 100, field < 100                │
│    - 大于等于/小于等于：field >= 100, field <= 100      │
│    - 包含：field in ['a', 'b', 'c']                     │
│    - 与/或：expr1 and expr2, expr1 or expr2             │
│                                                         │
│ 2. 性能优化建议                                          │
│    - 标量字段建立索引（Milvus 会自动处理）              │
│    - 避免在向量检索前做复杂标量过滤                     │
│    - 先用向量检索召回，再用标量过滤精简                 │
│                                                         │
│ 3. 常见应用场景                                          │
│    - 按类别/标签筛选内容                                │
│    - 按时间范围过滤（最近 7 天、最近 30 天）             │
│    - 按权限控制（只返回用户可访问的文档）               │
│    - 按状态筛选（只返回已发布/审核通过的文档）          │
│                                                         │
│ 4. 注意事项                                              │
│    - 字符串比较区分大小写                               │
│    - 空值处理：field != '' 或 field IS NOT NULL         │
│    - 特殊字符需要转义                                   │
│                                                         │
└─────────────────────────────────────────────────────────┘

RAG 系统中的典型用法:

场景：用户问"AI 相关的付费教程有哪些？"
策略：向量检索 + 标量过滤

```python
results = client.search(
    collection_name="knowledge_base",
    data=[query_vector],      # 问题的向量
    limit=5,
    filter="category == 'AI' and price > 0",  # 标量条件
    output_fields=["title", "content", "price"]
)
```
""")

scalar_query_best_practices()

## 总结

标量查询适合：
- 精确条件筛选
- 按类别、状态、日期等字段过滤
- 与向量检索结合使用（先语义召回，再标量过滤）

下一步：02_vector_search.py（向量检索）